In [89]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel , Field
from typing import Literal
from langchain_core.runnables import RunnableParallel,RunnableBranch,RunnableLambda



In [90]:
load_dotenv()

True

In [91]:
model = ChatGoogleGenerativeAI (
    model="gemini-2.5-flash",
    temperature=0,
)

In [92]:
parser = StrOutputParser()

In [93]:
class Feedback(BaseModel):
    sentiment: Literal['positive','Negative'] = Field(description='Give the sentiment of the feedback')

In [94]:
parser2 = PydanticOutputParser(pydantic_object = Feedback)

In [95]:
prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variable = ['feedback'],
    partial_variables = {'format_instruction': parser2.get_format_instructions()}

)

In [96]:
classifier_chain = prompt1 | model | parser2

In [97]:
# result = classifier_chain.invoke({'feedback' : 'This is a terrible smartphone'})

Here we can see that we are getting positive or clearly a single word but we do not have control over it ...
The LLM can also print "The sentiment is Positive/Negative"!!

We need to ensure that the output always remains as Negative/Positive basically a single word defining the sentiment 
So that it works smoothly when used as input to others.

In [98]:
# print(result) ## prints Negative or positive in a json format

In [99]:
prompt2 = PromptTemplate(
    template = 'Write an appropriate response to this positive feedback \n {feedback}',
    input_variables= ['feedback']
)

In [100]:
prompt3 = PromptTemplate(
    template = 'Write an appropriate response to this negative feedback \n {feedback}',
    input_variables= ['feedback']
)

In [101]:
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

In [102]:
chain = classifier_chain | branch_chain

In [105]:
print(chain.invoke({'feedback': 'This is a terrible phone'}))

could not find sentiment


In [104]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
